In [1]:
import Pkg

In [2]:
Pkg.activate(".")
Pkg.add("ThreadPinning")
Pkg.add("BenchmarkTools")
Pkg.add("ProfileCanvas")
Pkg.add("QuantumControl")
Pkg.add("QuantumPropagators")
Pkg.add("QuantumControlTestUtils")
Pkg.instantiate()

  Activating project at `~/Documents/KrylovKitBenchmark`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.t

In [3]:
using ThreadPinning
pinthreads(:cores)
threadinfo()

Hostname: 	cuny
CPU(s): 	2 x Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz
CPU target: 	cascadelake
Cores: 		32 (64 CPU-threads due to 2-way SMT)
NUMA domains: 	2 (16 cores each)

Julia threads: 	8

CPU socket 1
  0,32, 1,33, 2,34, 3,35, 4,36, 5,37, 6,38, 7,39, 
  8,40, 9,41, 10,42, 11,43, 12,44, 13,45, 14,46, 15,47

CPU socket 2
  16,48, 17,49, 18,50, 19,51, 20,52, 21,53, 22,54, 23,55, 
  24,56, 25,57, 26,58, 27,59, 28,60, 29,61, 30,62, 31,63


# = Julia thread, # = Julia thread on HT, # = >1 Julia thread

(Mapping: 1 => 0, 2 => 1, 3 => 2, 4 => 3, 5 => 4, ...)


In [4]:
filter(p -> contains(p[1], "THREAD"), ENV)

Dict{String, String} with 6 entries:
  "OPENBLAS_NUM_THREADS"   => "1"
  "VECLIB_MAXIMUM_THREADS" => "1"
  "OMP_NUM_THREADS"        => "1"
  "NUMEXPR_NUM_THREADS"    => "1"
  "MKL_NUM_THREADS"        => "1"
  "JULIA_NUM_THREADS"      => "8"

In [5]:
using QuantumControl: Trajectory
using QuantumControlTestUtils.DummyOptimization: dummy_control_problem

In [6]:
N = 100

100

In [7]:
problem = dummy_control_problem(; N, n_trajectories=Threads.nthreads(), n_steps=100, density=1.0, hermitian=true)
trajs = problem.trajectories;
tlist = problem.tlist;

In [8]:
using QuantumPropagators: Cheby
using QuantumControl: propagate_trajectories

In [9]:
propagate_trajectories(trajs, tlist; method=Cheby, use_threads=true);

In [10]:
using BenchmarkTools

In [13]:
bm_sequential = @benchmark propagate_trajectories($trajs, $tlist; method=Cheby, use_threads=false)

BenchmarkTools.Trial: 101 samples with 1 evaluation.
 Range (min … max):  46.467 ms … 59.764 ms  ┊ GC (min … max): 0.00% … 13.72%
 Time  (median):     48.786 ms              ┊ GC (median):    3.25%
 Time  (mean ± σ):   49.627 ms ±  2.253 ms  ┊ GC (mean ± σ):  3.37% ±  2.73%

      ▂ ▂ █▇█▂                                                 
  ▃▃▁▁█▆█▆█████▅█▆▆▃▃▅▁▃▃▁▃▆▆▃▅▃▅▆▅▆▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▃
  46.5 ms         Histogram: frequency by time        58.4 ms <

 Memory estimate: 17.41 MiB, allocs estimate: 25672.

In [14]:
bm_parallel = @benchmark propagate_trajectories($trajs, $tlist; method=Cheby, use_threads=true)

BenchmarkTools.Trial: 386 samples with 1 evaluation.
 Range (min … max):  10.425 ms … 19.436 ms  ┊ GC (min … max):  0.00% … 37.14%
 Time  (median):     13.571 ms              ┊ GC (median):    13.75%
 Time  (mean ± σ):   12.974 ms ±  1.603 ms  ┊ GC (mean ± σ):   9.96% ±  7.37%

    ▂▄▃▁▁▁                  ▄▄█▁▂▂▄                            
  ▂▃██████▆▅▃▄▃▃▃▁▂▁▁▁▂▁▃▅▄▇███████▆█▆▆▄▃▄▃▂▃▃▂▁▁▂▁▁▁▁▁▂▁▂▁▁▂ ▃
  10.4 ms         Histogram: frequency by time        17.3 ms <

 Memory estimate: 17.41 MiB, allocs estimate: 25714.

In [16]:
mean(bm_sequential.times) / mean(bm_parallel.times)

3.824971127132747